# This notebook estimates moment magnitude exceedance probability

for fault0 only, following two steps:
1. evaluate if a fault would slip
2. if yes, then estimate estimated Mw exceedance proability

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from tqdm import tqdm
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

################## User Inputs ############################## 
name_prefix = '251104'; n_cases = 86; n_faults = 12; n_times = 6; 
fault_id = 0
slip_threshold = 0.004 # for fault0 0.005 corresponds to at least 6 cells
n_samples = 100_000 # for Monte Carlo sampling
# fault_area = 16_985_344.51 # fault total area in m2, for fault0
Mw_target = 1.5 # target magnitude for exceedance probability
coor_fault_file_path = repo_root/'results'/'Sula_CCS'/'coor_fault'/'JD_Sula_2025_gmc_coor&fault.npy'
total_failed_cells_file_path = repo_root/'results'/'Sula_CCS'/'fault_slip_analysis'/f'{name_prefix}_total_failed_cells.npy'
parameters_file_path = repo_root/'data'/'Sula_CCS'/'params_responses'/f'{name_prefix}_CMG_parameters.csv'
fault_slip_analysis_folder_path = repo_root/'results'/'Sula_CCS'/'fault_slip_analysis'
################## End of User Inputs #######################

coor_fault_all = np.load(coor_fault_file_path)
total_failed_cells = np.load(total_failed_cells_file_path)
parameters = pd.read_csv(parameters_file_path)

# fault_cell_count = np.full((n_faults), np.nan)
fault_cell_count_all = np.full((n_faults), np.nan)
for fault_id_itr in range(0,n_faults):
    fault_id_mask_all = (coor_fault_all[:,:,:,3] == fault_id_itr)
    fault_cell_count_all[fault_id_itr] = np.count_nonzero(fault_id_mask_all)

failed_cell_ratio = total_failed_cells / fault_cell_count_all[np.newaxis,:,np.newaxis]

# estimate Mw through Monte Carlo sampling
# initialize an array to hold Mw exceedance probability for all cases and time steps
# Mw_exceedance = np.full((n_cases,n_times), np.nan) # array shape (n_cases,n_times)
Mw_exceedance = np.zeros((n_cases,n_times)) # array shape (n_cases,n_times)

mean_tau = np.load(fault_slip_analysis_folder_path/f'{name_prefix}_mean_tau.npy')

# run analysis
for case_num in tqdm(range(1,n_cases+1), desc='Estimating moment magnitude'):
    # extract the fault area from the parameter dataframe
    fault_area = parameters.loc[parameters["case_name"] == f'case{case_num}','A_m2'].iloc[0]

    for time_step in range(n_times):
        failure_ratio = failed_cell_ratio[case_num-1,fault_id,time_step]
        if failure_ratio >= slip_threshold:

            mean_shear_stress = mean_tau[case_num-1,fault_id,time_step]
            stress_drop = np.random.uniform(mean_shear_stress * 0.1, mean_shear_stress * 0.9, n_samples) * 1e6 # convert from MPa to Pa
            rupture_area = np.random.uniform(fault_area * failure_ratio, fault_area, n_samples)
            # calculate moment magnitude
            Mw = (np.log10(stress_drop * rupture_area ** 1.5) - 9.1) * 2 / 3

            # Kernel density for smooth PDF estimate
            kde = gaussian_kde(Mw)
            Mw_grid = np.linspace(Mw.min(), Mw.max(), 1000)
            pdf = kde(Mw_grid)

            # Empirical CDF
            Mw_sorted = np.sort(Mw)
            cdf = np.arange(1, len(Mw_sorted) + 1) / len(Mw_sorted)

            # Exceedance probability = 1 - CDF
            exceedance_prob = 1 - cdf

            # Find exceedance probability at target Mw
            Mw_exceedance[case_num-1,time_step] = np.mean(Mw > Mw_target)

# create headers
# Define column names in a list
column_headers = [f'time{i}' for i in range(n_times)]
# Join the list of names into a single string using your delimiter
header_string = ','.join(column_headers)
np.savetxt(fault_slip_analysis_folder_path/f'{name_prefix}_Mw_exceedance{Mw_target}_fault0.csv',Mw_exceedance,delimiter=',',fmt='%.4f',header=header_string,comments='')

# print(Mw_exceedance)

Estimating moment magnitude: 100%|██████████| 86/86 [00:22<00:00,  3.88it/s]


In [3]:
Mw_exceedance
print(np.nonzero(Mw_exceedance[:,2]))
print(np.count_nonzero(Mw_exceedance[:,2]))
Mw_exceedance[:,2]
# np.mean(Mw_exceedance[:,2])

(array([58, 60, 61, 66, 71, 72, 75, 76, 77, 79, 81, 83, 85]),)
13


array([0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.44299, 0.     , 0.30836, 0.3286 , 0.     ,
       0.     , 0.     , 0.     , 0.37644, 0.     , 0.     , 0.     ,
       0.     , 0.56926, 0.4332 , 0.     , 0.     , 0.52732, 0.26703,
       0.52288, 0.     , 0.26126, 0.     , 0.2237 , 0.     , 0.23391,
       0.     , 0.50518])

In [4]:
parameters = pd.read_csv(repo_root/'data'/'Sula_CCS'/'params_responses'/f'{name_prefix}_CMG_parameters.csv')
parameters['SH_azi_deg'].iloc[np.nonzero(Mw_exceedance[:,2])]

58    319.28
60    319.30
61    319.14
66    319.51
71    319.12
72    319.13
75    319.17
76    319.80
77    319.91
79    319.71
81    319.10
83    319.58
85    319.56
Name: SH_azi_deg, dtype: float64

## calculate the (number of cases with fault slip / total cases)

In [ ]:
import numpy as np
from pathlib import Path

name_prefix = '250922'
base_path = Path('.')

FSA_combined = np.load(base_path/'data'/f'{name_prefix}_FSA_combined.npy')
no_slip_count = np.sum(FSA_combined == 0, axis=0)
slip_probability = 1 - no_slip_count/FSA_combined.shape[0]
# np.savetxt(base_path/'data'/f'{save_file_prefix}_FSA_probability.csv',slip_probability,delimiter=",",fmt="%.4f")
# Save as CSV
df = pd.DataFrame(
    slip_probability,
    columns=[f"time_{t}" for t in range(n_times)]
)
df.insert(0, "fault_id", range(n_faults))

df.to_csv(base_path/'data'/f'{name_prefix}_FSA_probability.csv', index=False, float_format="%.4f")
df

# calculate responses for DGSA sensitivity analysis

stress ratio and Mw exceedance probability for fault0

In [ ]:
import numpy as np
from pathlib import Path

Mw_target = 1.5 # target magnitude for exceedance probability
name_prefix = '251023'; fault_id = [0]; year = 2050; year_list = [2030, 2040, 2050, 2060, 2550, 3050]
base_path = Path('.')
mean_stress_ratio = np.load(base_path/'data'/f'{name_prefix}_mean_stress_ratio.npy')
Mw_exceedance_prob = np.loadtxt(base_path/'data'/f'{name_prefix}_Mw_exceedance{Mw_target}_fault0.csv',delimiter=',',skiprows=1)
resp = mean_stress_ratio[:,fault_id,year_list.index(year)]
resp = np.insert(resp,1,Mw_exceedance_prob[:,year_list.index(year)],axis=1)

np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_fault{fault_id}.csv',resp,delimiter=',',fmt='%.4f',header=f'stress_ratio,exceedance{Mw_target}_prob',comments='')
print(mean_stress_ratio.shape)
print(Mw_exceedance_prob.shape)
resp